In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# -------------------------------------------------------------------
# 1. Load and concatenate all yearly CSVs
# -------------------------------------------------------------------

data_dir = Path("data")
csv_files = sorted(data_dir.glob("*.csv"))

df_per_year = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_file"] = f.name  # optional: track origin
    df_per_year.append(df)

df = pd.concat(df_per_year, ignore_index=True)
print("Rows:", len(df), "Files:", len(csv_files))

# df = pd.read_csv("data\\raw_traffic_road_accident_2017.csv")


# -------------------------------------------------------------------
# 2. Standardise column names used later
#    (adjust these if your actual column headers differ)
# -------------------------------------------------------------------

# Example columns from Kaggle/data.gov.uk description:
# Reference Number, Grid Ref: Easting, Grid Ref: Northing, Number of Vehicles,
# Accident Date, Time (24hr), 1st Road Class, Road Surface,
# Lighting Conditions, Weather Conditions, Casualty Severity, etc. [web:17][web:19]

col_rename = {
    "Reference Number": "accident_id",
    "Grid Ref: Easting": "easting",
    "Grid Ref: Northing": "northing",
    "Number of Vehicles": "num_vehicles",
    "Accident Date": "accident_date",
    "Time (24hr)": "time_24h",
    "1st Road Class": "road_class",
    "Road Surface": "road_surface",
    "Lighting Conditions": "lighting_conditions",
    "Weather Conditions": "weather_conditions",
    "Casualty Severity": "casualty_severity",
}

# Only rename columns that actually exist
col_rename = {k: v for k, v in col_rename.items() if k in df.columns}
df = df.rename(columns=col_rename)

# -------------------------------------------------------------------
# 3. Parse date and time into year, month, weekday, hour-of-day
# -------------------------------------------------------------------

# Parse accident_date (expected format like "09/01/2016")
df["accident_date"] = pd.to_datetime(
    df["accident_date"], errors="coerce", dayfirst=True
)

# Some files may not have a separate date column and encode year in Expr1
if df["accident_date"].isna().all() and "Expr1" in df.columns:
    # Fallback: try to extract year from Expr1 (e.g. 'Leeds 2016') and
    # create a fake date using that year plus a generic month/day
    year = (
        df["Expr1"]
        .astype(str)
        .str.extract(r"(\d{4})", expand=False)
        .astype("float").astype("Int64")
    )
    df["year"] = year
else:
    df["year"] = df["accident_date"].dt.year

# Time parsing: ensure we have strings like "1615", "08:30", etc.
df["time"] = df["time_24h"].astype(str).str.zfill(4)
df["hour"] = df["time"].str[:2].astype(int)

# Derive month and weekday (if we have valid dates)
df["month"] = df["accident_date"].dt.month
df["weekday"] = df["accident_date"].dt.day_name()

# -------------------------------------------------------------------
# 4. Create categorical bins
# -------------------------------------------------------------------

# Severity: if values are already like "Slight", "Serious", "Fatal", just keep them.
# We can optionally normalise case/spacing.
df["severity"] = (
    df["casualty_severity"]
    .astype(str)
    .str.strip()
    .str.title()
)

# Number of vehicles bin
if "num_vehicles" in df.columns:
    df["vehicles_bin"] = pd.cut(
        df["num_vehicles"].astype("float"),
        bins=[0, 1, 2, np.inf],
        labels=["1", "2", "3+"],
        right=True
    )

# Number of casualties bin (if available)
df["num_casualties"] = df.groupby(["accident_id"]).transform("size")


# Normalise some condition fields to nicer categories
for col in ["road_class", "road_surface", "weather_conditions", "lighting_conditions"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# -------------------------------------------------------------------
# 5. Keep numeric fields for spatial plots and speed/road class
# -------------------------------------------------------------------

# easting/northing are provided as integers; keep them as-is
# If you also have lat/lon columns you can keep them similarly.

# Optional: drop rows without valid coordinates if you only want mappable accidents
df["has_coords"] = (~df["easting"].isna()) & (~df["northing"].isna())

# Keep only rows with both easting and northing
df = df[df["has_coords"]]

# -------------------------------------------------------------------
# 6. Select and export tidy columns (one row per accident/casualty record)
# -------------------------------------------------------------------

cols_to_keep = [
    # identifiers / spatial
    "accident_id",
    "easting",
    "northing",
    "has_coords",
    # time
    "accident_date",
    "year",
    "month",
    "weekday",
    "hour",
    "time_24h",
    # core measures
    "num_vehicles",
    "vehicles_bin",
    "num_casualties" if "num_casualties" in df.columns else None,
    "casualties_bin" if "casualties_bin" in df.columns else None,
    # conditions
    "road_class" if "road_class" in df.columns else None,
    "road_surface" if "road_surface" in df.columns else None,
    "lighting_conditions" if "lighting_conditions" in df.columns else None,
    "weather_conditions" if "weather_conditions" in df.columns else None,
    "speed_limit" if "speed_limit" in df.columns else None,
    # outcomes
    "severity",
    "casualty_severity",
]

cols_to_keep = [c for c in cols_to_keep if c is not None and c in df.columns]
tidy = df[cols_to_keep].copy()

# Drop rows with completely missing key fields if needed
tidy = tidy.dropna(subset=["year", "hour", "severity"], how="any")

# -------------------------------------------------------------------
# 7. Save to a tidy CSV file for all three systems
# -------------------------------------------------------------------

output_path = Path("data\data.csv")
tidy.to_csv(output_path, index=False)

print(f"Saved tidy dataset with {len(tidy)} rows and {len(tidy.columns)} columns to {output_path.resolve()}")


Saved tidy dataset with 2203 rows and 18 columns to F:\Codespace\ivcw\data\data.csv


C:\Users\易彤\AppData\Local\Temp\ipykernel_22968\3211092556.py:45: UserWarning: Parsing dates in %m/%d/%Y format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["accident_date"] = pd.to_datetime(
